# Sentinel-2 seleccionado por calidad y por año → Amazon S3

Selecciona un año de **2020 a 2025**. Dentro de cada mes se intenta conservar **una adquisición real por recuadro** en cada bloque: 1–7, 8–14, 15–21 y 22–fin. Se mantienen tus 49 números de partida y el filtro territorial del 85 %.

Se evalúa primero SCL a 20 m; las diez bandas solo se transfieren después de elegir la mejor observación. Los bloques sin datos suficientes quedan registrados como faltantes. No se inventan imágenes ni se mezclan días para completar semanas.

Los TIFF se construyen en RAM de Colab y se guardan en S3; no ocupan el disco de tu computadora. Los planes de cada año quedan separados y la caché de calidad en S3 permite reanudar una sesión nueva.

**Usa CPU.** Este notebook no necesita ZIP. Sustituye solo `sentinel_s3.py` y este notebook; conserva tus otros archivos y tu configuración.

Ejecuta dentro de los límites de tu cuenta y plan. Colab prohíbe utilizar varias cuentas para eludir sus límites; este selector organiza tareas por año y no automatiza cuentas ni sesiones.

## 1. Cargar archivos sueltos del proyecto

Primera subida: selecciona estos cinco archivos de la raíz de tu carpeta actual:

- `sentinel_s3.py`
- `cobertura_territorial.py`
- `datos_sentinel.py` (ya lo tienes; no se ha cambiado)
- `utilidades.py` (ya lo tienes; no se ha cambiado)
- `configuracion.json` (conserva el tuyo)

Segunda subida: selecciona `michoacan_inegi_original.json`, que ya tienes dentro de `Delimitacion_rendimiento_aguacate/fuentes`.

Esta celda carga código y geometrías existentes; no carga imágenes satelitales ni requiere comprimir tu proyecto.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, sys, json, importlib

RAIZ = Path("/content/proyecto_sentinel_s3")
RAIZ.mkdir(parents=True, exist_ok=True)
requeridos = ["sentinel_s3.py", "cobertura_territorial.py", "datos_sentinel.py", "utilidades.py", "configuracion.json"]
print("Selecciona los cinco archivos de la raíz de tu proyecto:", ", ".join(requeridos))
subidos = files.upload()
faltan = set(requeridos) - set(subidos)
if faltan:
    raise ValueError("Faltan archivos: " + ", ".join(sorted(faltan)) + ". Selecciona los archivos sueltos, sin comprimir.")
for nombre in requeridos:
    (RAIZ / nombre).write_bytes(subidos[nombre])
del subidos
print("Ahora selecciona michoacan_inegi_original.json, de Delimitacion_rendimiento_aguacate/fuentes.")
subidos = files.upload()
nombre = "michoacan_inegi_original.json"
if nombre not in subidos:
    raise ValueError("Selecciona michoacan_inegi_original.json de la carpeta fuentes.")
fuentes = RAIZ / "Delimitacion_rendimiento_aguacate" / "fuentes"
fuentes.mkdir(parents=True, exist_ok=True)
(fuentes / nombre).write_bytes(subidos[nombre])
del subidos
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26,<3", "pandas>=2.2,<3", "rasterio>=1.4.3,<2", "shapely>=2.1,<3",
    "pyproj>=3.6,<4", "requests>=2.32,<3", "boto3>=1.40,<2", "matplotlib>=3.8,<4"], check=True)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
for modulo in ("sentinel_s3", "cobertura_territorial", "datos_sentinel", "utilidades"):
    sys.modules.pop(modulo, None)
importlib.invalidate_caches()
print("Proyecto cargado desde archivos sueltos. Se conserva tu configuración.")

## 2. Subir la cuadrícula y sus metadatos

Sube `cortes_georreferenciados.csv` y `metadatos.json` de tu carpeta `datos/.../sentinel2`. También se admite `plan_recuadros.csv`. Si subes los dos CSV, se elige automáticamente `cortes_georreferenciados.csv`.

Se aceptan los sufijos de copia `(1)`, `(2)` que pueda añadir Colab. Estos archivos aportan las coordenadas de los mismos números del mapa.

In [ ]:
subidos = files.upload()
import re

def nombre_normalizado(nombre):
    return re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(nombre).name).lower()

nombre_csv = next((n for n in subidos if nombre_normalizado(n) == "cortes_georreferenciados.csv"), None)
if nombre_csv is None:
    nombre_csv = next((n for n in subidos if nombre_normalizado(n) == "plan_recuadros.csv"), None)
if nombre_csv is None:
    raise ValueError("Sube cortes_georreferenciados.csv o plan_recuadros.csv de la cuadrícula original.")
nombre_meta = next((n for n in subidos if nombre_normalizado(n) == "metadatos.json"), None)
ENTRADAS = RAIZ / "entradas_s3"
ENTRADAS.mkdir(exist_ok=True)
RUTA_MALLA = ENTRADAS / "cortes_georreferenciados.csv"
RUTA_META = ENTRADAS / "metadatos.json"
RUTA_MALLA.write_bytes(subidos[nombre_csv])
if nombre_meta:
    RUTA_META.write_bytes(subidos[nombre_meta])
else:
    RUTA_META.unlink(missing_ok=True)
print("Se utilizará:", nombre_csv)
print("Si subiste ambos CSV, se prefirió cortes_georreferenciados.csv.")
print("Cuadrícula recibida; las imágenes satelitales todavía no se han transferido.")

## 3. Elegir año, bucket y criterios

- `ANIO_DESCARGA`: el año completo que procesará esta ejecución, entre 2020 y 2025.
- `COBERTURA_MINIMA_AOI`: cobertura municipal para conservar un recuadro; 0.85 como antes.
- `FRACCION_VALIDA_MINIMA`: al menos 95 % del recuadro debe estar cubierto por datos válidos de esa adquisición.
- `FRACCION_CLARA_MINIMA`: al menos 80 % del recuadro debe estar libre de nubes/sombras según las clases SCL aceptadas.

Son criterios iniciales configurables. Revisa los faltantes por temporada antes de decidir si conviene ajustarlos. El programa nunca los rebaja automáticamente.

Escribe tu bucket y región; si ya están en tu configuración, deja los campos vacíos para conservarlos. La configuración original de tu computadora no se reemplaza.

In [ ]:
ANIO_DESCARGA = 2020 #@param [2020, 2021, 2022, 2023, 2024, 2025] {type:"raw"}
BUCKET = "" #@param {type:"string"}
REGION = "" #@param {type:"string"}
COBERTURA_MINIMA_AOI = 0.85 #@param {type:"number"}
FRACCION_VALIDA_MINIMA = 0.95 #@param {type:"number"}
FRACCION_CLARA_MINIMA = 0.80 #@param {type:"number"}

CONFIG = RAIZ / "configuracion.json"
from sentinel_s3 import configuracion, preparar, planificar, transferir
cfg = configuracion(CONFIG, anio=ANIO_DESCARGA)
cfg.pop("_raiz", None)
cfg.pop("_trabajo", None)
cfg["territorio"]["fraccion_minima_aoi"] = float(COBERTURA_MINIMA_AOI)
cfg["sentinel_s3"].update(
    bucket=BUCKET.strip() or cfg["sentinel_s3"]["bucket"],
    region=REGION.strip() or cfg["sentinel_s3"]["region"],
    archivo_malla=str(RUTA_MALLA), archivo_metadatos_malla=str(RUTA_META),
    carpeta_trabajo="resultados_sentinel_s3", anio_descarga=ANIO_DESCARGA,
    seleccion_temporal={"modo": "mejor_por_bloque", "fraccion_valida_minima": float(FRACCION_VALIDA_MINIMA),
                        "fraccion_clara_minima": float(FRACCION_CLARA_MINIMA), "usar_cache_s3": True})
CONFIG.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
TRABAJO = configuracion(CONFIG)["_trabajo"]
print("Año:", ANIO_DESCARGA)
print("Periodo:", cfg["sentinel_s3"]["fecha_inicial"], "a", cfg["sentinel_s3"]["fecha_final"])
print("Carpeta de este año:", TRABAJO)
print("Recuadros de partida:", cfg["territorio"]["cortes_preseleccionados"])
print("Después de cambiar año o criterios, vuelve a ejecutar las secciones 4 y 6 antes de transferir.")

## 4. Preparar y ver los recuadros filtrados

Esta celda restringe la malla original a los 49 números de tu selección y después calcula el filtro con los polígonos oficiales. No accede a S3 ni descarga bandas. Revisa el mapa y la cobertura municipal que queda después de descartar los bordes.

In [ ]:
from IPython.display import display, Image
import pandas as pd

seleccion = preparar(configuracion(CONFIG))
TRABAJO = configuracion(CONFIG)["_trabajo"]
display(Image(filename=str(TRABAJO / "mapa_recuadros_filtrados.png")))
display(pd.read_csv(TRABAJO / "recuadros_seleccionados.csv")[["corte", "porcentaje_aoi"]])
display(pd.read_csv(TRABAJO / "cobertura_municipal_tras_filtro.csv"))
print((TRABAJO / "informe_filtro.json").read_text(encoding="utf-8"))

## 5. Conectar AWS antes de evaluar calidad

Guarda las claves en **Secrets**, icono de llave de Colab, y permite su acceso a este notebook. Nombres: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` y, solo para sesiones temporales, `AWS_SESSION_TOKEN`. Si no usas Secrets, se pedirán con entrada oculta.

No escribas las claves en código ni las compartas en el chat. Si son temporales, marca `CREDENCIALES_TEMPORALES`. Si vencen, actualiza los secretos y repite esta celda antes de reanudar.

La planificación siguiente guarda metadatos de calidad en tu bucket para reanudar. Por eso ahora se conecta AWS antes de planificar. Las claves permanecen privadas y no se guardan en los planes.

In [ ]:
import os
from getpass import getpass
from google.colab import userdata

CREDENCIALES_TEMPORALES = False #@param {type:"boolean"}
def leer_secreto(nombre, obligatorio=True):
    try:
        valor = userdata.get(nombre)
    except Exception:
        valor = ""
    if not valor and obligatorio:
        valor = getpass(nombre + " (entrada oculta): ")
    return valor.strip() if valor else ""

access_key = leer_secreto("AWS_ACCESS_KEY_ID")
secret_key = leer_secreto("AWS_SECRET_ACCESS_KEY")
es_temporal = CREDENCIALES_TEMPORALES or access_key.startswith("ASIA")
session_token = leer_secreto("AWS_SESSION_TOKEN", obligatorio=es_temporal) if es_temporal else ""
if not access_key or not secret_key or (es_temporal and not session_token):
    raise ValueError("Faltan credenciales. Para sesiones temporales se necesitan tres valores.")
os.environ["AWS_ACCESS_KEY_ID"] = access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = secret_key
if session_token:
    os.environ["AWS_SESSION_TOKEN"] = session_token
else:
    os.environ.pop("AWS_SESSION_TOKEN", None)
del access_key, secret_key, session_token
print("Credenciales disponibles en memoria; no se guardaron en el proyecto.")

## 6. Seleccionar las mejores adquisiciones del año

Esta etapa consulta el catálogo del año, lee **solo SCL y su máscara de validez** para comparar candidatos y guarda la caché de metadatos en S3. **Todavía no transfiere las diez bandas.** Puede tardar; muestra avances por recuadro y fecha.

Para cada bloque y recuadro, elige la adquisición con mayor fracción clara entre las que cumplen los mínimos. En empate, prefiere mayor validez y cercanía al centro del bloque. Las fechas reales se conservan.

Si una misma toma necesita varias teselas para cubrir el recuadro, las evalúa conjuntamente y las guarda después como fragmentos nativos de un solo conjunto. Nunca fusiona fechas distintas.

Se generan `cobertura_temporal.csv` e `informe_seleccion_temporal.json`. Si hay errores de lectura, el plan no permite transferir; repite esta sección para reintentar únicamente las evaluaciones pendientes. Si reinicias la sesión, renueva credenciales y se recuperará la caché de S3.

In [ ]:
plan = planificar(configuracion(CONFIG))
print("Conjuntos seleccionados:", plan["conjuntos_periodo_recuadro"])
print("Bloques faltantes:", plan["resumen_temporal"]["faltantes"])
print("COG de bandas previstos:", plan["archivos_banda_previstos"])
print("GiB estimados sin comprimir:", plan["gib_nativos_sin_comprimir_estimados"])
cobertura = pd.read_csv(TRABAJO / "cobertura_temporal.csv")
display(cobertura.groupby(["mes", "estado"]).size().unstack(fill_value=0))
display(cobertura.head(20))

## 7. Probar un conjunto seleccionado

Transfiere un conjunto periodo–recuadro, hasta 1 GiB nuevo. Un conjunto contiene una adquisición; si cruza teselas puede tener varios fragmentos, cada uno con diez bandas y SCL.

Se reutilizan los TIFF ya guardados que coincidan con la adquisición elegida. Al terminar, busca el manifiesto `_completo.json` en `aguacate/sentinel2/selecciones/identificador_del_plan/`. Los datos nativos permanecen bajo `aguacate/sentinel2/nativo/`.

`motivo_parada: max_casos` es normal para la prueba. `errores` debe estar vacío. Esta etapa sí guarda bandas en S3 y puede generar cargos.

In [ ]:
estado_prueba = transferir(configuracion(CONFIG), max_casos=1, max_gib=1)
if estado_prueba["errores"]:
    raise RuntimeError("La prueba registró errores. Revisa ultima_transferencia.json antes de continuar.")

## 8. Completar el año elegido

Activa `EJECUTAR_ANIO_COMPLETO`. Solo se transferirán las adquisiciones seleccionadas del año indicado en la sección 3. `MAX_GIB_POR_EJECUCION = 0` intenta completar todo el año; un valor como 20 permite avanzar por partes.

El estado `terminado: true` significa que se transfirieron los conjuntos seleccionados; los bloques sin adquisición o sin calidad suficiente siguen registrados como faltantes. No significa que haya 48 observaciones completas para cada recuadro.

Los archivos previos de S3 no se borran. Si un archivo ya descargado corresponde a la selección nueva, se reutiliza. Para otro año, cambia el selector de la sección 3 y vuelve a preparar y planificar.

In [ ]:
EJECUTAR_ANIO_COMPLETO = False #@param {type:"boolean"}
MAX_GIB_POR_EJECUCION = 0 #@param {type:"number"}

if EJECUTAR_ANIO_COMPLETO:
    estado = transferir(configuracion(CONFIG), max_casos=0, max_gib=float(MAX_GIB_POR_EJECUCION))
    if estado["errores"]:
        raise RuntimeError("Hay conjuntos pendientes por error; revisa el informe y vuelve a ejecutar.")
    print("Año:", ANIO_DESCARGA, "· transferencia completada:", estado["terminado"])
else:
    print("Activa EJECUTAR_ANIO_COMPLETO después de revisar el plan y la prueba.")

## Reanudar o cambiar de año

Para reanudar en una sesión nueva, vuelve a cargar los mismos archivos sueltos y la malla, elige el mismo año y criterios, conecta AWS y ejecuta preparar/planificar/transferir. La caché SCL se recupera de S3 y los TIFF completos se reutilizan.

Para otro año, cambia `ANIO_DESCARGA` en la sección 3. Cada año usa una carpeta de trabajo distinta y un identificador de plan propio. Utiliza una sola configuración por ejecución; no cambies el año mientras una celda siga trabajando.

Los datos mantienen la resolución nativa, la calibración y las fechas originales. Las nuevas capas del modelo todavía no se entrenan en este notebook.